# ⏰ Notebook 1: The Fixed-Timeout Problem

Heartbeat-based failure detectors traditionally use a **single timeout**: 'if I don't hear from you within `T` seconds, you're dead'. Picking `T` is a no-win:

- **Too short** → frequent false positives during normal network jitter.
- **Too long** → real failures take ages to notice.

Let's feel the pain on a noisy network.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/phi-accrual-failure-detection
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 Simulate a noisy heartbeat stream

In [ ]:
import random
random.seed(7)
INTERVAL = 1.0      # node sends a heartbeat every ~1 second
JITTER = 0.4        # ± noisy network
TOTAL = 30.0
DEAD_AT = 20.0      # node really dies at t=20

now = 0.0
heartbeats = []
while now < TOTAL:
    now += INTERVAL + random.uniform(-JITTER, JITTER)
    if now >= DEAD_AT:
        break
    heartbeats.append(now)
print(f'received {len(heartbeats)} heartbeats; node truly dies at t={DEAD_AT}')


In [ ]:
def fixed_timeout_alarms(beats, timeout, total=TOTAL, step=0.1):
    alarms = []
    last = 0.0
    i = 0
    t = 0.0
    state = 'up'
    while t < total:
        while i < len(beats) and beats[i] <= t:
            last = beats[i]; i += 1
            if state == 'down':
                state = 'up'
        if t - last > timeout and state == 'up':
            alarms.append((round(t,2), 'DOWN'))
            state = 'down'
        t += step
    return alarms

for to in [0.5, 1.5, 3.0]:
    a = fixed_timeout_alarms(heartbeats, to)
    print(f'timeout={to}s -> {len(a)} alarm(s); first at {a[0] if a else None}')


Short timeouts panic on jitter; long timeouts are slow. We want a detector whose **suspicion grows smoothly** as silence stretches, learns the network's normal cadence, and lets the *application* pick how paranoid to be.

👉 Next notebook: **phi accrual** failure detector.